In [16]:
# Cell 1: 导入库
from langchain.agents import create_agent  # LangChain 1.0 的 create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.runnables import RunnablePassthrough
# new for langchain 1.0
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
import os

In [17]:
# Cell 2: 定义工具
@tool
def add(a: float, b: float) -> float:
    """计算两个数的和"""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """计算两个数的乘积"""
    return a * b

tools = [add, multiply]

In [18]:
# Cell 3: 配置 LLM
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com",
    temperature=0
)

In [19]:
# Cell 4: 创建带 Memory 的 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手。你可以使用提供的工具来回答问题。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [20]:
# Cell 5: 创建 Agent（LangChain 1.0 风格）
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个有帮助的助手。",
)

In [22]:
# Cell 6: 手动管理 Memory（LangChain 1.0 推荐方式）
# 在 1.0 中，Memory 不再内置在 AgentExecutor 中，而是手动维护
from langchain_core.messages import HumanMessage, AIMessage

# 初始化 chat_history
chat_history = []

In [23]:
# Cell 7: 封装调用函数
def chat_with_memory(user_input: str):
    global chat_history
    
    # 构建消息列表
    messages = []
    
    # 添加系统消息
    messages.append(("system", "你是一个有帮助的助手。"))
    
    # 添加历史对话
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            messages.append(("user", msg.content))
        else:
            messages.append(("assistant", msg.content))
    
    # 添加当前用户输入
    messages.append(("user", user_input))
    
    # 调用 Agent
    response = agent.invoke({"messages": messages})
    
    # 提取 AI 回复
    ai_response = response["messages"][-1].content
    
    # 更新记忆
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=ai_response))
    
    return ai_response

In [24]:
# Cell 8: 测试多轮对话
print("第一轮：")
result1 = chat_with_memory("我叫张三")
print(f"回答: {result1}")

print("\n第二轮：")
result2 = chat_with_memory("我叫什么名字？")
print(f"回答: {result2}")

print("\n第三轮：")
result3 = chat_with_memory("3 + 4 等于多少？")
print(f"回答: {result3}")

print("\n第四轮：")
result4 = chat_with_memory("刚才那个结果再乘以 2 是多少？")
print(f"回答: {result4}")

# 打印完整对话历史
print("\n完整对话历史：")
for msg in chat_history:
    role = "用户" if isinstance(msg, HumanMessage) else "助手"
    print(f"{role}: {msg.content}")

第一轮：
回答: 你好张三！很高兴认识你。我是一个有帮助的助手，可以帮你进行数学计算，比如加法和乘法。有什么我可以帮助你的吗？

第二轮：
回答: 你刚才告诉我你叫张三。

第三轮：
回答: 3 + 4 = 7

第四轮：
回答: 7 × 2 = 14

完整对话历史：
用户: 我叫张三
助手: 你好张三！很高兴认识你。我是一个有帮助的助手，可以帮你进行数学计算，比如加法和乘法。有什么我可以帮助你的吗？
用户: 我叫什么名字？
助手: 你刚才告诉我你叫张三。
用户: 3 + 4 等于多少？
助手: 3 + 4 = 7
用户: 刚才那个结果再乘以 2 是多少？
助手: 7 × 2 = 14


## 基础理解
### Agent 的 Memory 是什么？它和 RAG 的知识库有什么不同？
- memory相当于人的记忆，会记录之前的问题与回答。它是会不停地变化的，而rag会是固定的。
### ConversationBufferMemory 和 ConversationSummaryMemory 有什么区别？各适合什么场景？
- ConversationBufferMemory会原封不动地把之前的问答记录下来，而ConversationSummaryMemory会总结然后记录。前者适用于对话历史比较短的场景，后者适用于长段对话，因为比较省内存。
### 在 prompt 中，MessagesPlaceholder(variable_name="chat_history") 的作用是什么？
- 这样会让每次的query中都使用之前的记录。
## 动手验证
### 把 ConversationBufferMemory 换成 ConversationSummaryMemory，多轮对话后输出有什么变化？
- 精准度会有所下降。
### 如果 Memory 不设置 input_key="input"，代码会报什么错？为什么需要这个参数？
- 补充：会报 KeyError: 'input'，因为 Agent 不知道哪个参数是用户输入
### 在多轮对话中问 "刚才那个结果再乘以 2 是多少？"，Agent 如何知道“刚才那个结果”是什么？
- 通过调取memory中之前的问答。
## 深入思考
### Memory 会无限增长吗？如何限制 Memory 的大小？如果不限制会有什么问题？
- memory会一直增长，直到遇到内存上限。可以在代码中定义memory的大小。不限制则会内存overflow。
### 如果 Agent 同时需要“长期记忆”（用户偏好）和“短期记忆”（当前对话），架构上如何设计？
- 长期记忆和短期记忆可以分开来定义，将长期定义固定写在system_prompt中，而短期记忆可以设置大小上限来不停地更新。
### 你今天的 Agent 只能记住对话历史。如果要让它记住用户偏好（如“我喜欢简洁的回答”），需要做什么扩展？
- 需要定义在system信息中。